# 5. Advanced RAG 기법 — Agent 자율 검색 판단 + Context Memory

Agent가 검색 전략을 자율적으로 결정하는 **Agentic RAG**와 대화 맥락을 유지하는 **Context Memory**를 학습합니다.

| 파트 | 주제 | 핵심 내용 |
|------|------|-----------|
| **Part 1** | Agent 자율 검색 판단 | Naive RAG 한계 → GradeDocuments → StateGraph 재검색 루프 |
| **Part 2** | Context Memory | Context Engineering 매트릭스 → 메모리 3유형 → InMemoryStore |


## 0) 환경 설정
> 진행: [▓░░░░░░░░░] 1/11

In [1]:
# [5-1] : 라이브러리 설치
# [핵심] Agentic RAG + Context Memory에 필요한 전체 패키지 — 첫 실행 시 한 번만 필요
# [주의] 버전 충돌 시 런타임 재시작 후 재실행

!uv pip install -q langchain langchain-core langchain-openai langchain-chroma chromadb langchain-text-splitters langgraph python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


D:\2026_04_SK하이닉스_AI에이전트프레임워크구현\배포버전\.venv\Scripts\python.exe: No module named pip


In [2]:
# [5-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True → 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."

# 트레이싱 설정 (선택)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

print(f"OPENAI_API_KEY: {'OK' if OPENAI_API_KEY else 'MISSING'} (필수)")

OPENAI_API_KEY: OK (필수)


In [3]:
# [5-3] : 공통 LLM 및 임베딩 모델 초기화
# [핵심] 노트북 전체에서 재사용할 모델을 한 곳에서 관리
# [주의] temperature=0 → 결정적 출력; gpt-4.1-mini로 비용 절감

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536차원

print(f"LLM: {llm.model_name}")
print(f"Embedding: {embedding.model}")

LLM: gpt-4.1-mini
Embedding: text-embedding-3-small


---
# Part 1: Agent 자율 검색 판단

Naive RAG의 한계를 분석하고, Agent가 검색 전략을 자율적으로 결정하는 **Agentic RAG** 아키텍처를 구축한다.


## 1) Naive RAG 한계 분석

# [1-1] : Naive RAG 한계 분석

**Naive RAG**는 가장 기본적인 검색 증강 생성 파이프라인이다: `질문 → 검색(top-k) → LLM 생성`. 그러나 실무에서 다음과 같은 한계가 드러난다:

| 한계 | 설명 |
|------|------|
| **검색 정밀도 부족** | 임베딩 유사도만으로 top-k를 선택 — 의미적으로 관련 없는 문서가 포함됨 |
| **관련성 검증 부재** | 검색 결과의 품질을 평가하지 않고 그대로 LLM에 전달 |
| **재검색 불가** | 검색 결과가 부적절해도 한 번의 검색으로 끝남 — 자기 수정 능력 없음 |
| **고정 쿼리** | 사용자 질문을 그대로 검색 쿼리로 사용 — 검색에 최적화된 쿼리 변환 없음 |
| **환각(Hallucination)** | 관련 없는 문서 기반으로 그럴듯한 답변을 생성 |

### Naive RAG vs Agentic RAG

| 단계 | Naive RAG | Agentic RAG |
|------|-----------|-------------|
| **검색 결정** | 항상 검색 | Agent가 필요 여부를 자율 판단 |
| **쿼리 생성** | 원본 질문 그대로 | Agent가 검색에 최적화된 쿼리 생성 |
| **관련성 평가** | 없음 | GradeDocuments로 문서 품질 검증 |
| **재검색** | 불가 | 관련성 낮으면 쿼리 재작성 후 재검색 |
| **생성** | 전체 문서 전달 | 관련 문서만 선별 전달 |

> 진행: [▓▓░░░░░░░░] 2/11

## 2) Agentic RAG 아키텍처

# [1-2] : Agentic RAG 아키텍처

**Agentic RAG**는 Agent가 검색 전략을 자율적으로 결정하는 아키텍처이다. 핵심은 검색 결과를 **평가**하고, 부적절할 경우 **재검색**하는 자기 수정 루프이다.

### 핵심 컴포넌트

| 컴포넌트 | 역할 | 구현 방식 |
|---------|------|----------|
| **Retriever** | 벡터스토어에서 문서 검색 | `similarity_search(query, k=3)` |
| **GradeDocuments** | 검색 문서의 관련성 평가 | Pydantic + `with_structured_output` |
| **Query Rewriter** | 관련성 낮을 때 쿼리 재작성 | LLM 기반 쿼리 변환 |
| **Re-ranker** | 검색 결과 재정렬 | LLM 기반 관련성 점수 |
| **StateGraph** | 전체 흐름 제어 | 조건 분기 + 순환 엣지 |

### 아키텍처 흐름

```
START → gen_query → [tools_condition] → retrieve → grade_documents
                              ↓ __end__              ↓ relevant
                             END           generate_answer → END
                                                ↓ not_relevant
                                          rewrite_query → gen_query
```

> 진행: [▓▓▓░░░░░░░] 3/11

In [4]:
# [5-4] : 벡터스토어 구축 — 샘플 문서 + ChromaDB
# [핵심] Agentic RAG의 검색 기반이 되는 벡터스토어를 구축
# [주의] persist_directory="./chroma_advanced_rag_db"로 로컬 파일 기반 DB를 생성하며, reset_directory=True로 재실행 시 중복을 방지

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from notebook_vectorstore import build_vectorstore_from_documents

raw_docs = [
    Document(
        page_content="RAG(Retrieval-Augmented Generation)는 외부 지식베이스에서 관련 문서를 검색하여 LLM의 답변 정확도를 높이는 패턴이다. "
                     "LLM의 두 가지 핵심 제약인 유한한 컨텍스트 창과 지식 마감일 문제를 해결한다. "
                     "검색 단계에서 임베딩 유사도로 top-k 문서를 추출하고, 생성 단계에서 LLM이 해당 문서를 참고하여 답변한다.",
        metadata={"topic": "RAG 개요", "source": "ai-tech", "category": "retrieval"}
    ),
    Document(
        page_content="Agentic RAG는 기존 RAG의 한계를 극복한 아키텍처로, LLM 에이전트가 검색 전략을 자율적으로 결정한다. "
                     "단순 단일 검색이 아닌 반복 검색, 쿼리 재작성, 다중 소스 검색 등을 에이전트가 동적으로 선택한다. "
                     "관련성 평가 → 필요 시 재검색 → 최종 답변 생성의 루프를 통해 품질을 높인다.",
        metadata={"topic": "Agentic RAG", "source": "ai-tech", "category": "retrieval"}
    ),
    Document(
        page_content="LangGraph는 LLM으로 상태 기반 멀티 액터 애플리케이션을 구축하기 위한 프레임워크이다. "
                     "StateGraph로 노드(처리 단계)와 엣지(전이 조건)를 정의하고, 조건부 엣지로 복잡한 라우팅 로직을 구현한다. "
                     "순환 그래프를 지원하여 반복 검색이나 자기 수정 루프를 자연스럽게 표현할 수 있다.",
        metadata={"topic": "LangGraph", "source": "ai-tech", "category": "framework"}
    ),
    Document(
        page_content="Context Memory는 대화 맥락을 세션 간에 유지하는 패턴이다. "
                     "InMemoryStore를 사용하여 사용자별 네임스페이스에 정보를 저장하고, "
                     "시맨틱 검색으로 현재 대화와 관련된 과거 맥락을 동적으로 검색한다. "
                     "Semantic(프로필/사실), Episodic(과거 경험), Procedural(규칙) 세 유형의 메모리를 활용한다.",
        metadata={"topic": "Context Memory", "source": "ai-tech", "category": "memory"}
    ),
    Document(
        page_content="임베딩(Embedding)은 텍스트를 고차원 벡터 공간에 표현하는 기술로, 의미적으로 유사한 텍스트는 벡터 공간에서도 가깝게 위치한다. "
                     "OpenAI의 text-embedding-3-small은 1536차원 벡터를 생성하며 RAG 시스템의 검색 품질을 결정한다. "
                     "코사인 유사도나 내적(Inner Product)으로 쿼리와 문서 간 유사도를 측정한다.",
        metadata={"topic": "임베딩", "source": "ai-tech", "category": "retrieval"}
    ),
    Document(
        page_content="GradeDocuments는 검색된 문서의 관련성을 LLM이 자동 평가하는 패턴이다. "
                     "Pydantic BaseModel과 with_structured_output을 결합하여 "yes" 또는 "no" 라벨을 구조화 출력으로 생성한다. "
                     "관련 없는 문서는 버리고 쿼리를 재작성하여 재검색하는 Self-RAG 루프의 핵심 컴포넌트이다.",
        metadata={"topic": "GradeDocuments", "source": "ai-tech", "category": "evaluation"}
    ),
    Document(
        page_content="Re-ranking은 1차 검색된 문서를 LLM이 질문과의 관련성을 기준으로 재정렬하는 기법이다. "
                     "임베딩 유사도 검색은 빠르지만 정밀도가 낮을 수 있으므로, Re-ranking으로 정밀도를 보완한다. "
                     "Cross-encoder나 LLM 기반 점수 매기기로 구현하며, 검색 후보 수(top-k)를 적절히 제한해야 비용을 관리할 수 있다.",
        metadata={"topic": "Re-ranking", "source": "ai-tech", "category": "retrieval"}
    ),
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(raw_docs)

CHROMA_DIR = "./chroma_advanced_rag_db"
COLLECTION_NAME = "agentic_rag_05"
vectorstore = build_vectorstore_from_documents(
    documents=splits,
    embedding=embedding,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    reset_directory=True,
)

print(f"청킹 결과: {len(raw_docs)}개 문서 → {len(splits)}개 청크")
print(f"벡터스토어 구축 완료: {CHROMA_DIR} | 컬렉션: {COLLECTION_NAME}")

from langchain_core.tools import tool

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """AI/LLM 기술 지식베이스에서 관련 문서를 검색합니다."""
    docs = vectorstore.similarity_search(query, k=3)
    serialized = "\n\n".join(
        f"[{i+1}] topic={d.metadata.get('topic')}\n{d.page_content}"
        for i, d in enumerate(docs)
    )
    return serialized, docs

print("retrieve 도구 등록 완료")


청킹 결과: 7개 문서 → 7개 청크
벡터스토어 구축 완료: ./chroma_advanced_rag_db | 컬렉션: agentic_rag_05
retrieve 도구 등록 완료


## 3) 문서 관련성 평가 (GradeDocuments)

# [1-3] : 문서 관련성 평가 (GradeDocuments)

**GradeDocuments**는 검색된 문서의 관련성을 LLM이 자동으로 평가하는 핵심 패턴이다. `with_structured_output`으로 구조화된 `yes/no` 판정을 받아 조건 분기에 사용한다.

| 판정 | 의미 | 다음 행동 |
|------|------|-----------|
| `yes` | 문서가 질문과 관련 있음 | → `generate_answer` (답변 생성) |
| `no` | 문서가 질문과 관련 없음 | → `rewrite_query` (쿼리 재작성 후 재검색) |

> 진행: [▓▓▓▓░░░░░░] 4/11

In [5]:
# [5-5] : GradeDocuments — Pydantic 스키마 + LLM 관련성 평가
# [핵심] Literal["yes", "no"]로 관련성을 이진 분류 — StateGraph 조건 분기의 기준
# [주의] with_structured_output은 JSON mode 강제 — 파싱 에러 최소화

from pydantic import BaseModel, Field
from typing import Literal


class GradeDocuments(BaseModel):
    """검색된 문서의 관련성 평가 결과."""
    score: Literal["yes", "no"] = Field(
        description="문서가 질문과 관련 있으면 'yes', 없으면 'no'"
    )
    reasoning: str = Field(description="관련성 판단 근거 — 한 문장")


grader = llm.with_structured_output(GradeDocuments)

# 테스트: 관련 있는 문서
test_result = grader.invoke(
    "질문: RAG에서 문서 관련성을 평가하는 방법은?\n\n"
    "문서: GradeDocuments는 검색된 문서의 관련성을 LLM이 자동 평가하는 패턴이다.\n\n"
    "이 문서가 질문과 관련이 있습니까?"
)
print(f"관련성 평가: score={test_result.score} | {test_result.reasoning}")

# 테스트: 관련 없는 문서
test_result2 = grader.invoke(
    "질문: RAG에서 문서 관련성을 평가하는 방법은?\n\n"
    "문서: 오늘 날씨가 좋으니 산책을 가자.\n\n"
    "이 문서가 질문과 관련이 있습니까?"
)
print(f"관련성 평가: score={test_result2.score} | {test_result2.reasoning}")

관련성 평가: score=yes | 질문은 RAG에서 문서 관련성을 평가하는 방법에 대해 묻고 있으며, 문서에서는 GradeDocuments라는 패턴이 검색된 문서의 관련성을 LLM이 자동으로 평가하는 방법임을 설명하고 있다. 따라서 문서 내용이 질문과 직접적으로 관련되어 있다.


관련성 평가: score=no | 질문은 RAG에서 문서 관련성을 평가하는 방법에 관한 것이며, 제공된 문서는 '오늘 날씨가 좋으니 산책을 가자'라는 내용으로 질문과 직접적인 관련성이 없습니다.


## 4) 쿼리 재작성 (Query Rewrite)

# [1-4] : 쿼리 재작성 (Query Rewrite)

검색된 문서가 관련 없을 때, 원래 질문을 더 구체적으로 **재작성**하여 검색 품질을 향상시킨다. 이것이 Agentic RAG의 핵심인 **자기 수정 루프**이다.

```
원본: "RAG의 성능을 높이는 방법은?"
  → 재작성: "RAG 시스템에서 검색 정밀도를 높이기 위한 Re-ranking과 Query Decomposition 기법"
```

> 진행: [▓▓▓▓▓░░░░░] 5/11

In [6]:
# [5-6] : 쿼리 재작성 함수 정의
# [핵심] LLM이 원본 질문을 검색에 최적화된 형태로 변환
# [주의] 재작성된 쿼리가 원본 의도를 유지하도록 프롬프트 설계가 중요

from langchain_core.messages import HumanMessage


def rewrite_query(question: str) -> str:
    """검색 품질 향상을 위해 질문을 재작성합니다."""
    prompt = (
        f"다음 질문을 다른 용어와 관점으로 재작성하여 검색 품질을 높이세요.\n"
        f"원래 의미는 유지하되, 더 구체적인 키워드를 포함하세요.\n\n"
        f"원본 질문: {question}\n재작성:"
    )
    rewritten = llm.invoke(prompt)
    return rewritten.content


# 테스트
original_q = "RAG의 성능을 높이는 방법은?"
rewritten_q = rewrite_query(original_q)

print(f"원본: {original_q}")
print(f"재작성: {rewritten_q}")

원본: RAG의 성능을 높이는 방법은?
재작성: RAG(Retrieval-Augmented Generation) 모델의 정확도와 효율성을 향상시키는 최적화 기법은 무엇인가요?


## 5) Agent 자율 판단 StateGraph

# [1-5] : Agent 자율 판단 StateGraph

LangGraph `StateGraph`로 Agentic RAG 파이프라인을 구현한다. 핵심은 `grade_documents` 노드의 결과에 따라 **generate**(답변 생성) 또는 **rewrite**(쿼리 재작성 후 재검색)로 분기하는 **조건부 엣지**이다.

| 노드 | 역할 |
|------|----- |
| `gen_query` | 도구 호출 또는 직접 응답 결정 |
| `retrieve` | 벡터스토어 검색 실행 (ToolNode) |
| `grade_documents` | 검색 결과 관련성 평가 (yes/no) |
| `generate_answer` | 관련 문서 기반 최종 답변 생성 |
| `rewrite_query` | 관련 없을 때 질문 재작성 후 재검색 루프 |

> 진행: [▓▓▓▓▓▓░░░░] 6/11

In [7]:
# [5-7] : StateGraph 상태 정의 + 노드 함수 구현
# [핵심] RAGState는 그래프 전체에서 공유되는 상태 — relevance 필드로 라우팅 결정
# [주의] generate_query_or_respond는 llm.bind_tools로 도구 호출 능력을 부여받은 LLM 사용

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, AIMessage
from langgraph.graph.message import add_messages


class RAGState(TypedDict):
    """Agentic RAG의 공유 상태."""
    messages: Annotated[Sequence[BaseMessage], add_messages]  # 메시지 히스토리
    relevance: str  # "yes" 또는 "no" — grade_documents 결과


llm_with_tools = llm.bind_tools([retrieve])


def generate_query_or_respond(state: RAGState):
    """검색 쿼리를 생성하거나 직접 응답합니다 — 진입 노드."""
    response = llm_with_tools.invoke(state["messages"], config=lf_config)
    return {"messages": [response]}


def grade_documents_node(state: RAGState):
    """검색된 문서의 관련성을 평가합니다."""
    msgs = state["messages"]
    user_q = next((m.content for m in msgs if isinstance(m, HumanMessage)), "")
    tool_content = next(
        (m.content for m in reversed(msgs) if hasattr(m, "type") and m.type == "tool"),
        ""
    )
    grade = grader.invoke(
        f"질문: {user_q}\n\n검색된 문서:\n{tool_content}\n\n이 문서들이 질문과 관련이 있습니까?"
    )
    print(f"  [관련성 평가] score={grade.score} | {grade.reasoning}")
    return {"relevance": grade.score, "messages": msgs}


def generate_answer_node(state: RAGState):
    """관련 문서를 바탕으로 최종 답변을 생성합니다."""
    msgs = state["messages"]
    user_q = next((m.content for m in msgs if isinstance(m, HumanMessage)), "")
    tool_content = next(
        (m.content for m in reversed(msgs) if hasattr(m, "type") and m.type == "tool"),
        ""
    )
    prompt = (
        f"다음 문서만 참고하여 질문에 답변하세요.\n"
        f"문서:\n{tool_content}\n\n질문: {user_q}"
    )
    resp = llm.invoke(prompt, config=lf_config)
    return {"messages": [resp]}


def rewrite_query_node(state: RAGState):
    """검색 품질 향상을 위해 질문을 재작성합니다."""
    user_q = next(
        (m.content for m in state["messages"] if isinstance(m, HumanMessage)), ""
    )
    rewritten = rewrite_query(user_q)
    print(f"  [쿼리 재작성] {rewritten[:80]}...")
    return {"messages": [HumanMessage(content=rewritten)]}


print("노드 함수 4개 정의 완료")
print("  - generate_query_or_respond (진입)")
print("  - grade_documents_node (관련성 평가)")
print("  - generate_answer_node (답변 생성)")
print("  - rewrite_query_node (쿼리 재작성)")

노드 함수 4개 정의 완료
  - generate_query_or_respond (진입)
  - grade_documents_node (관련성 평가)
  - generate_answer_node (답변 생성)
  - rewrite_query_node (쿼리 재작성)


In [8]:
# [5-8] : StateGraph 조립 — 노드 등록 및 엣지 연결
# [핵심] add_conditional_edges로 grade_documents 결과에 따라 generate 또는 rewrite 분기
# [주의] rewrite_query → gen_query 순환 엣지가 자기 수정 루프를 구현 — 무한 루프 방지 필요

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def relevance_router(state: RAGState) -> str:
    """관련성 평가 결과에 따라 다음 노드를 결정합니다."""
    return "generate" if state.get("relevance") == "yes" else "rewrite"


workflow = StateGraph(RAGState)

# 노드 등록
workflow.add_node("gen_query", generate_query_or_respond)
workflow.add_node("retrieve", ToolNode([retrieve]))
workflow.add_node("grade_documents", grade_documents_node)
workflow.add_node("generate_answer", generate_answer_node)
workflow.add_node("rewrite_query", rewrite_query_node)

# 엣지 연결
workflow.add_edge(START, "gen_query")
workflow.add_conditional_edges(
    "gen_query",
    tools_condition,  # 도구 호출이면 retrieve, 아니면 END
    {"tools": "retrieve", "__end__": END},
)
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    relevance_router,  # yes → generate, no → rewrite
    {"generate": "generate_answer", "rewrite": "rewrite_query"},
)
workflow.add_edge("rewrite_query", "gen_query")  # 재검색 루프
workflow.add_edge("generate_answer", END)

graph_app = workflow.compile()
print("Agentic RAG StateGraph 컴파일 완료")

Agentic RAG StateGraph 컴파일 완료


In [9]:
# [5-9] : Agentic RAG StateGraph 실행 테스트
# [핵심] stream()으로 각 노드의 실행 순서를 실시간 확인 — 조건 분기 흐름 시각화
# [주의] 관련성이 'no'이면 rewrite_query → gen_query → retrieve 루프 발생

test_q = "임베딩 벡터의 유사도 측정 방법과 RAG에서의 역할을 설명해주세요."

print(f"질문: {test_q}\n")
print("=== 그래프 실행 흐름 ===")

final_answer = ""
for event in graph_app.stream(
    {"messages": [{"role": "user", "content": test_q}]},
):
    for node_name, output in event.items():
        print(f"  [{node_name}] 실행")
        if node_name == "generate_answer" and "messages" in output:
            final_answer = output["messages"][-1].content

print(f"\n=== 최종 답변 ===")
print(final_answer[:500])

질문: 임베딩 벡터의 유사도 측정 방법과 RAG에서의 역할을 설명해주세요.

=== 그래프 실행 흐름 ===


  [gen_query] 실행

=== 최종 답변 ===



## 6) Re-ranking으로 검색 품질 강화

# [1-6] : Re-ranking으로 검색 품질 강화

**Re-ranking**은 1차 검색된 문서를 LLM이 질문과의 관련성을 기준으로 **재정렬**하는 기법이다. Agent 판단 흐름 내에서 검색 정밀도를 높이는 데 활용된다.

| 단계 | 방식 | 장점 | 단점 |
|------|------|------|------|
| **1차 검색** | 벡터 유사도 (top-k) | 빠름, 대규모 후보 추출 | 정밀도 낮음 |
| **Re-ranking** | LLM 기반 점수 매기기 | 높은 정밀도 | 추가 비용/지연 |

> 진행: [▓▓▓▓▓▓▓░░░] 7/11

In [10]:
# [5-10] : LLM 기반 Re-ranker 구현 + 검색 품질 비교
# [핵심] 검색 결과 각각에 대해 LLM이 관련성 점수(0~10)를 부여하여 재정렬
# [주의] Re-ranking은 LLM 호출 비용이 추가됨 — 후보 수(top-k)를 적절히 제한


class RelevanceScore(BaseModel):
    """문서-질문 간 관련성 점수."""
    score: int = Field(description="관련성 점수 (0~10)", ge=0, le=10)
    reasoning: str = Field(description="점수 부여 이유")


reranker_llm = llm.with_structured_output(RelevanceScore)


def rerank_documents(query: str, documents: list[Document], top_n: int = 3) -> list[dict]:
    """검색 결과를 LLM 기반으로 재정렬한다."""
    scored = []
    for doc in documents:
        prompt = (
            f"질문: {query}\n\n"
            f"문서: {doc.page_content}\n\n"
            f"이 문서가 질문에 얼마나 관련이 있는지 0~10 점수로 평가하세요."
        )
        result = reranker_llm.invoke(prompt)
        scored.append({"doc": doc, "score": result.score, "reasoning": result.reasoning})
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_n]


# Re-ranking 적용 전/후 비교
rerank_query = "Agentic RAG에서 검색 품질을 높이는 방법"
candidates = vectorstore.similarity_search(rerank_query, k=5)
reranked = rerank_documents(rerank_query, candidates, top_n=3)

print(f"=== 1차 검색 (top-5) ===")
for i, doc in enumerate(candidates):
    print(f"  [{i+1}] topic={doc.metadata.get('topic')} | {doc.page_content[:60]}...")

print(f"\n=== Re-ranking 후 (top-3) ===")
for i, item in enumerate(reranked):
    doc = item["doc"]
    print(f"  [{i+1}] score={item['score']}/10 | topic={doc.metadata.get('topic')}")
    print(f"      이유: {item['reasoning'][:60]}...")

=== 1차 검색 (top-5) ===
  [1] topic=Agentic RAG | Agentic RAG는 기존 RAG의 한계를 극복한 아키텍처로, LLM 에이전트가 검색 전략을 자율적으로 결...
  [2] topic=RAG 개요 | RAG(Retrieval-Augmented Generation)는 외부 지식베이스에서 관련 문서를 검색하여 ...
  [3] topic=Re-ranking | Re-ranking은 1차 검색된 문서를 LLM이 질문과의 관련성을 기준으로 재정렬하는 기법이다. 임베딩 유...
  [4] topic=임베딩 | 임베딩(Embedding)은 텍스트를 고차원 벡터 공간에 표현하는 기술로, 의미적으로 유사한 텍스트는 벡터 ...
  [5] topic=GradeDocuments | GradeDocuments는 검색된 문서의 관련성을 LLM이 자동 평가하는 패턴이다. Pydantic Bas...

=== Re-ranking 후 (top-3) ===
  [1] score=9/10 | topic=Agentic RAG
      이유: 문서는 Agentic RAG의 검색 품질을 높이는 구체적인 방법들(반복 검색, 쿼리 재작성, 다중 소스 검색...
  [2] score=8/10 | topic=Re-ranking
      이유: 문서에서는 Re-ranking 기법을 통해 1차 검색된 문서의 정밀도를 높이는 방법을 설명하고 있으며, 이는...
  [3] score=8/10 | topic=GradeDocuments
      이유: 문서에서는 Agentic RAG의 핵심 컴포넌트인 GradeDocuments 패턴을 설명하며, 검색된 문서의...


---
# Part 2: Context Memory

Agent의 자율 판단 능력에 더해, **대화 맥락을 유지**하는 Context Memory를 학습한다. Context Engineering의 체계적 프레임워크와 메모리 3유형을 이해하고, `InMemoryStore`로 구현한다.


## 7) Context Engineering 개요

# [2-1] : Context Engineering 개요

**Context Engineering**은 "올바른 정보를, 올바른 형식으로, 올바른 시점에" AI에 제공하는 시스템 설계이다. 단순한 프롬프트 엔지니어링을 넘어, 컨텍스트를 **런타임에 동적으로** 구성하는 아키텍처 접근법이다.

### 2차원 매트릭스 (가변성 x 수명)

| | **단기 (Within Session)** | **장기 (Cross Session)** |
|---|---|---|
| **정적 (Static)** | 시스템 프롬프트, 도구 정의 | 사용자 프로필, 글로벌 규칙 |
| **동적 (Dynamic)** | 대화 히스토리, 검색 결과 | 메모리(Semantic/Episodic/Procedural) |

- **단기 정적**: 한 세션 내에서 변하지 않는 정보 (시스템 프롬프트, 도구 스키마)
- **단기 동적**: 대화가 진행되며 변하는 정보 (메시지 히스토리, RAG 검색 결과)
- **장기 정적**: 세션을 초월하여 유지되는 고정 정보 (사용자 프로필, 도메인 규칙)
- **장기 동적**: 세션을 초월하며 계속 업데이트되는 정보 (학습된 선호도, 과거 경험)

> 진행: [▓▓▓▓▓▓▓▓░░] 8/11

## 8) 메모리 3유형

# [2-2] : 메모리 3유형

인지과학에서 영감을 받은 세 가지 메모리 유형으로 Agent의 장기 기억을 분류한다. 각 유형에 따라 **저장 구조**와 **활용 방식**이 다르다.

| 메모리 유형 | 역할 | 내용 예시 | Namespace 패턴 |
|------------|------|----------|---------------|
| **Semantic** | 사실/프로필 | 사용자 역할, 선호도, 도메인 지식 | `(user_id, "profile")` |
| **Episodic** | 과거 경험 | 이전 대화 요약, 문제 해결 경험 (few-shot) | `(user_id, "episodes")` |
| **Procedural** | 행동 규칙 | 응답 포맷, 도메인 가이드라인 | `(user_id, "procedures")` |

### 활용 패턴
- **Semantic**: 항상 로드 — 사용자가 누구인지 파악
- **Episodic**: 시맨틱 검색으로 관련 경험만 선별 — few-shot 프롬프트에 활용
- **Procedural**: 항상 로드 — 응답 규칙과 가이드라인 적용

> 진행: [▓▓▓▓▓▓▓▓▓░] 9/11

In [11]:
# [5-11] : InMemoryStore 기본 API — put / get / search
# [핵심] InMemoryStore는 (namespace, key) 쌍으로 데이터를 저장 — 사용자별 격리
# [주의] InMemoryStore는 프로세스 재시작 시 초기화 — 영구 저장은 PostgresStore 등 사용

# [2-3] : InMemoryStore + Namespace 패턴

from langgraph.store.memory import InMemoryStore

# 기본 스토어 (키-값 검색)
basic_store = InMemoryStore()
user_id = "user_skh_001"

# 사용자 선호도 저장
basic_store.put((user_id, "preferences"), "response_style", {"value": "상세하고 예시 포함"})
basic_store.put((user_id, "preferences"), "language", {"value": "ko"})
basic_store.put((user_id, "profile"), "role", {"value": "AI 엔지니어", "domain": "RAG 시스템"})

# 정확 키 조회
style_item = basic_store.get((user_id, "preferences"), "response_style")
print(f"응답 스타일: {style_item.value}")

# 네임스페이스 전체 검색
all_prefs = basic_store.search((user_id, "preferences"))
print(f"\n저장된 선호도 {len(all_prefs)}개:")
for item in all_prefs:
    print(f"  [{item.key}] = {item.value}")

응답 스타일: {'value': '상세하고 예시 포함'}

저장된 선호도 2개:
  [response_style] = {'value': '상세하고 예시 포함'}
  [language] = {'value': 'ko'}


In [12]:
# [5-12] : 시맨틱 검색 InMemoryStore — 임베딩 기반 메모리 검색
# [핵심] index={"embed": embedding, "dims": 1536} 설정으로 시맨틱 검색 활성화
# [주의] dims는 임베딩 모델의 출력 차원과 일치해야 함 — text-embedding-3-small은 1536

# [2-4] : 시맨틱 검색 기반 메모리 조회

# 시맨틱 검색 스토어 (임베딩 기반)
semantic_store = InMemoryStore(
    index={"embed": embedding, "dims": 1536}
)
ns = (user_id, "memories")

# 다양한 기억 저장
semantic_store.put(ns, "mem1", {"content": "RAG 시스템 구축 시 청크 크기를 500자로 설정했을 때 성능이 좋았다"})
semantic_store.put(ns, "mem2", {"content": "ChromaDB가 FAISS보다 메타데이터 필터링이 편리하다고 느꼈다"})
semantic_store.put(ns, "mem3", {"content": "좋아하는 점심 메뉴는 비빔밥"})
semantic_store.put(ns, "mem4", {"content": "LangGraph의 조건부 엣지 개념을 이해하는 데 어려움을 느꼈다"})
semantic_store.put(ns, "mem5", {"content": "temperature=0 설정이 RAG 답변의 일관성을 높인다는 것을 배웠다"})

print(f"메모리 5개 저장 완료 (namespace={ns})")

# 시맨틱 검색 — RAG 관련 기억
query1 = "벡터스토어와 검색 설정"
results1 = semantic_store.search(ns, query=query1, limit=2)
print(f"\n쿼리: '{query1}'")
for r in results1:
    print(f"  [{r.key}] {r.value['content']}")

# 시맨틱 검색 — LangGraph 관련 기억
query2 = "LangGraph 그래프 구현"
results2 = semantic_store.search(ns, query=query2, limit=2)
print(f"\n쿼리: '{query2}'")
for r in results2:
    print(f"  [{r.key}] {r.value['content']}")

메모리 5개 저장 완료 (namespace=('user_skh_001', 'memories'))



쿼리: '벡터스토어와 검색 설정'
  [mem1] RAG 시스템 구축 시 청크 크기를 500자로 설정했을 때 성능이 좋았다
  [mem2] ChromaDB가 FAISS보다 메타데이터 필터링이 편리하다고 느꼈다

쿼리: 'LangGraph 그래프 구현'
  [mem4] LangGraph의 조건부 엣지 개념을 이해하는 데 어려움을 느꼈다
  [mem1] RAG 시스템 구축 시 청크 크기를 500자로 설정했을 때 성능이 좋았다


In [13]:
# [5-13] : 메모리 3유형 통합 구축 — Semantic / Episodic / Procedural
# [핵심] 세 가지 메모리 유형을 각각 다른 namespace에 저장 — 역할별 분리
# [주의] Episodic은 과거 경험(few-shot)이므로 구체적 사례를 저장해야 효과적

mem_store = InMemoryStore(index={"embed": embedding, "dims": 1536})
uid = "user_skh_001"

# Semantic Memory — 사용자 프로필/사실 (단일 JSON)
mem_store.put((uid, "profile"), "main", {
    "name": "김엔지니어",
    "role": "AI 엔지니어",
    "domain": "RAG 시스템, LangGraph",
    "preferred_language": "Python",
})

# Episodic Memory — 과거 대화 경험 (다중 문서)
mem_store.put((uid, "episodes"), "ep1", {
    "content": "사용자가 ChromaDB 설치 문제를 물어봤을 때 pip install langchain-chroma chromadb로 해결",
    "timestamp": "2026-04-07",
})
mem_store.put((uid, "episodes"), "ep2", {
    "content": "RAG 청크 크기 질문에서 chunk_size=500, overlap=50 권장했을 때 만족함",
    "timestamp": "2026-04-07",
})

# Procedural Memory — 행동 규칙
mem_store.put((uid, "procedures"), "rules", {
    "content": "항상 코드 예시와 함께 설명. 핵심 개념에 볼드 표시. 한국어로 응답.",
})

print("메모리 3유형 저장 완료")
print(f"  Semantic: (uid, 'profile') → 사용자 프로필")
print(f"  Episodic: (uid, 'episodes') → 과거 대화 경험 2개")
print(f"  Procedural: (uid, 'procedures') → 응답 규칙")

# Episodic 시맨틱 검색
ep_results = mem_store.search((uid, "episodes"), query="벡터스토어 문제 해결", limit=1)
print(f"\n관련 과거 경험: {ep_results[0].value['content'] if ep_results else '없음'}")

메모리 3유형 저장 완료
  Semantic: (uid, 'profile') → 사용자 프로필
  Episodic: (uid, 'episodes') → 과거 대화 경험 2개
  Procedural: (uid, 'procedures') → 응답 규칙



관련 과거 경험: 사용자가 ChromaDB 설치 문제를 물어봤을 때 pip install langchain-chroma chromadb로 해결


## 9) Agentic RAG + Context Memory 통합

# [2-5] : Agentic RAG + Context Memory 통합

앞서 구현한 **Agentic RAG StateGraph**와 **Context Memory**를 통합한다. 매 대화마다 메모리에서 관련 맥락을 검색하여 시스템 프롬프트에 주입하고, 대화 경험을 에피소드 메모리에 저장한다.

| 단계 | 설명 |
|------|------|
| 1. 메모리 검색 | 프로필(항상) + 에피소드(시맨틱 검색) + 규칙(항상) |
| 2. 문서 검색 | 벡터스토어에서 관련 문서 top-k 검색 |
| 3. 컨텍스트 결합 | 메모리 + 문서를 시스템 프롬프트에 주입 |
| 4. 답변 생성 | LLM이 풍부한 맥락으로 답변 |
| 5. 경험 저장 | 대화 내용을 에피소드 메모리에 기록 |

> 진행: [▓▓▓▓▓▓▓▓▓▓] 10/11

In [14]:
# [5-14] : Context Memory + Agentic RAG 통합 — 멀티턴 대화
# [핵심] 매 대화마다 메모리에서 관련 맥락을 검색하여 시스템 프롬프트에 주입
# [주의] 메모리 검색 결과가 너무 많으면 컨텍스트 창을 초과 — limit 적절히 설정

from langchain_core.messages import SystemMessage
import time


def build_memory_context(store: InMemoryStore, user_id: str, query: str) -> str:
    """현재 쿼리와 관련된 메모리를 검색하여 컨텍스트를 구성합니다."""
    # 프로필 로드 (항상 포함)
    profile = store.get((user_id, "profile"), "main")
    profile_str = f"사용자 프로필: {profile.value}" if profile else ""

    # 관련 에피소드 검색 (시맨틱)
    episodes = store.search((user_id, "episodes"), query=query, limit=2)
    ep_str = "\n".join(
        f"  - {ep.value['content']}" for ep in episodes
    ) if episodes else "  없음"

    # 절차 규칙 로드
    rules = store.get((user_id, "procedures"), "rules")
    rules_str = f"응답 규칙: {rules.value['content']}" if rules else ""

    return f"{profile_str}\n\n관련 과거 경험:\n{ep_str}\n\n{rules_str}"


def memory_rag_chat(store: InMemoryStore, user_id: str, question: str) -> str:
    """Context Memory를 활용한 멀티턴 RAG 대화."""
    # 1) 메모리에서 관련 맥락 수집
    memory_ctx = build_memory_context(store, user_id, question)

    # 2) 벡터스토어에서 관련 문서 검색
    docs = vectorstore.similarity_search(question, k=3)
    doc_ctx = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

    # 3) 메모리 + 문서 컨텍스트를 결합하여 LLM 호출
    system_prompt = (
        f"다음 사용자 정보와 문서를 참고하여 답변하세요.\n\n"
        f"[사용자 맥락]\n{memory_ctx}\n\n"
        f"[참고 문서]\n{doc_ctx}"
    )
    response = llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=question)],
        config=lf_config,
    )

    # 4) 대화 경험을 에피소드 메모리에 저장
    ep_key = f"ep_{int(time.time())}"
    store.put((user_id, "episodes"), ep_key, {
        "content": f"질문: {question[:50]} → 답변 완료",
        "timestamp": "2026-04-13",
    })

    return response.content


# 멀티턴 대화 시뮬레이션
conversations = [
    "RAG에서 청크 크기를 어떻게 설정해야 하나요?",
    "제가 이전에 사용한 설정으로 ChromaDB를 구성하는 방법을 알려주세요.",
]

for turn, q in enumerate(conversations, 1):
    print(f"\n=== 대화 {turn}: {q} ===")
    answer = memory_rag_chat(mem_store, uid, q)
    print(answer[:350])


=== 대화 1: RAG에서 청크 크기를 어떻게 설정해야 하나요? ===


RAG(Retrieval-Augmented Generation)에서 **청크(chunk) 크기 설정**은 매우 중요합니다. 청크 크기는 문서를 쪼개는 단위로, 너무 크면 임베딩 품질이 떨어지고, 너무 작으면 문맥이 잘리지 않아 검색 효율이 떨어질 수 있습니다.

### 권장 청크 크기 설정
- **chunk_size=500 tokens** 정도가 일반적으로 적당합니다.
- **overlap=50 tokens** 정도로 겹치게 설정하면 문맥 연결이 자연스러워집니다.

이 설정은 문서의 의미 단위가 잘 유지되면서도, 임베딩과 검색 시 효율적인 벡터 생성에 도움을 줍니다.

### Python 예시 코드 (LangChain 

=== 대화 2: 제가 이전에 사용한 설정으로 ChromaDB를 구성하는 방법을 알려주세요. ===


김엔지니어님, 이전에 사용하신 **ChromaDB 설치 및 구성 방법**을 Python 코드 예시와 함께 설명드리겠습니다.

---

### 1. ChromaDB 설치

이전 경험에 따라, 다음 명령어로 설치하셨습니다:

```bash
pip install langchain-chroma chromadb
```

---

### 2. ChromaDB 구성 예시 (Python)

```python
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings

# 1) 임베딩 모델 초기화 (예: OpenAI 임베딩)
emb


---

# [정리] : 핵심 요약

| 항목 | 내용 |
|------|------|
| **다룬 기술** | Agentic RAG, GradeDocuments, Re-ranking, Context Memory, InMemoryStore |
| **핵심 개념** | Agent가 검색 전략을 자율 결정 + 대화 맥락을 메모리로 유지 |
| **GradeDocuments** | `with_structured_output`으로 관련성을 yes/no로 분류 — 조건 분기 기준 |
| **StateGraph 재검색 루프** | `rewrite_query → gen_query` 순환 엣지 — 자기 수정 루프 |
| **Re-ranking** | LLM 기반 점수 매기기로 검색 정밀도 강화 |
| **Context Engineering** | 2차원 매트릭스 (가변성 x 수명) — 올바른 정보를 올바른 시점에 제공 |
| **메모리 3유형** | Semantic(사실) / Episodic(경험) / Procedural(규칙) |
| **InMemoryStore** | `(namespace, key)` 패턴 + 시맨틱 검색 — 사용자별 메모리 격리 |

**Part 1 핵심 — Agentic RAG**
- Naive RAG의 한계 (관련성 검증 부재, 재검색 불가)를 `GradeDocuments` + `StateGraph` 재검색 루프로 극복
- `add_conditional_edges`로 관련성 평가 결과에 따라 동적 라우팅

**Part 2 핵심 — Context Memory**
- Context Engineering 2차원 매트릭스로 컨텍스트의 가변성과 수명을 체계적으로 분류
- `InMemoryStore` + namespace 패턴으로 Semantic/Episodic/Procedural 메모리를 구현
- 시맨틱 검색으로 현재 대화와 관련된 과거 기억을 동적으로 검색

→ 이를 발전시켜 **06_멀티에이전트** — 여러 Agent가 협력하는 아키텍처로 확장